In [11]:
!pip install --upgrade imbalanced-learn

In [12]:
import pandas as pd
import numpy as np
import math
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score
)
from sklearn.model_selection import train_test_split
from imblearn.over_sampling import SMOTE

# === Step 1: Load Data ===
file_path = "GPT_evaluation_corrected.xlsx"  # Change to your actual file path
df = pd.read_excel(file_path)

# === Step 2: Log-Normalize Metric Scores ===
features = ['BLEU', 'ROUGE-L', 'BERTScore']
target = 'Human_Judge'
epsilon = 1e-8
df_log = df.copy()
for feature in features:
    df_log[feature] = df_log[feature].apply(lambda x: math.log(x + epsilon))

# === Step 3: Prepare Features and Labels ===
X = df_log[features]
y = df_log[target]

# === Step 4: Train-Test Split with stratification ===
X_train, X_val, y_train, y_val = train_test_split(
    X, y, stratify=y, test_size=0.3, random_state=42
)

# === Step 5: Apply SMOTE on training data ===
sm = SMOTE(random_state=42)
X_train_res, y_train_res = sm.fit_resample(X_train, y_train)

# === Step 6: Train Logistic Regression on balanced data ===
model = LogisticRegression(solver='liblinear', random_state=42)
model.fit(X_train_res, y_train_res)

# === Step 7: Predict Probabilities on validation set ===
probs = model.predict_proba(X_val)[:, 1]

# === Step 8: Optimize threshold to maximize macro F1-score ===
best_thresh = 0.0
best_f1 = 0.0
thresholds = np.linspace(0, 1, 200)
for t in thresholds:
    preds = (probs >= t).astype(int)
    f1 = f1_score(y_val, preds, average='macro')  # macro balances classes
    if f1 > best_f1:
        best_f1 = f1
        best_thresh = t

# === Step 9: Final predictions with best threshold ===
final_preds = (probs >= best_thresh).astype(int)

# === Step 10: Output model coefficients and evaluation metrics ===
coeffs = dict(zip(features, model.coef_[0]))
intercept = model.intercept_[0]

print("Logistic Regression Coefficients:")
for feat, val in coeffs.items():
    print(f" {feat}: {val:.4f}")
print(f" Intercept: {intercept:.4f}\n")

print(f"Best threshold for classification: {best_thresh:.3f}")
print(f"Accuracy at best threshold: {(final_preds == y_val).mean():.4f}\n")

print("📊 Classification Report:\n", classification_report(y_val, final_preds))
print("📉 Confusion Matrix:\n", confusion_matrix(y_val, final_preds))
print("📈 AUC-ROC Score:", roc_auc_score(y_val, probs))


ImportError: cannot import name '_OneToOneFeatureMixin' from 'sklearn.base' (/Users/aijiazhou/opt/anaconda3/lib/python3.8/site-packages/sklearn/base.py)